In [10]:
import glob
import numpy as np
import os
import sys

# add parent folder (production) to sys.path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from utils.LoRa import MultiBAMv4

import importlib
import utils.my_lora_utils
importlib.reload(utils.my_lora_utils)
from utils.my_lora_utils import *

print(utils.my_lora_utils.__file__)

def parse_gt_from_filename(f):
    parts = f.split("_")
    return parts[6]

c:\Users\priba\Sean-2025\INC-LAB\BAM\INC-BAM\utils\my_lora_utils.py


In [11]:
sf = 9       # Spreading Factor 
N = 2**sf
input_row = 512
input_col = 33

In [ ]:
dataset_name_folder = f"dataset_sf{sf}_{input_row}x{input_col}"
dataset_clean_name_folder = f"clean_dataset_sf{sf}_{input_row}x{input_col}"
input_layer = input_row * input_col
second_layer = 2048
final_layer = 256
layers = [input_layer, second_layer , final_layer]  # compress 3840 → 1024 → 256

GEENRATE_ = False #### Secure accidently running

################### Load all .npy files ########################################
print("LOAD DATASET")
files = glob.glob(f'{dataset_name_folder}/*.npy')
data_list = []
gt_list = []
database_clean_signal = []
for f in files:
    x = np.load(f)
    gt_symbol = parse_gt_from_filename(f)
    data_list.append(x.flatten())
    gt_list.append(int(gt_symbol)) 

for sym in range(N): # 0 until 2**sf
    file_str = f'{dataset_clean_name_folder}/s_sf{sf}_bw125_{sym}.npy'
    x = np.load(file_str)
    x_flat= x.flatten()
    database_clean_signal.append(x_flat)
    
X = np.array(data_list)
gt_list = np.array(gt_list)
database_clean_signal = np.array(database_clean_signal)

################### Load all .npy files ########################################

multi_bam = MultiBAMv4(layers_dims=layers, eta=1e-5)

if (GEENRATE_):
    
    folder_path = f"weight_{input_layer}_{second_layer}_{final_layer}"

    # Check if folder exists, if not create it
    check_and_make_folder(folder_path)
    layer_losses = multi_bam.train(X=X, Y_sym = gt_list, database=database_clean_signal, num_epochs=10, batch_size=32)
    
    for i, bam in enumerate(multi_bam.bams):
        np.save(f"{folder_path}/weights_layer_{i}.npy", bam.W)

def load_weight(): 
    ## HOW TO LOAD WEIGHT
    layers = [input_row * input_col, 1024, 256] # <-- must match training

    multi_bam = MultiBAMv4(layers_dims=layers, eta=1e-5)
    for i, bam in enumerate(multi_bam.bams):
        bam.W = np.load(f"weight/weights_layer_{i}.npy")


LOAD DATASET
Folder created: weight_16896_2048_256

--- Training Layer 1/2 ---
EEpoch 1/10, MSE=0.011180
EEpoch 2/10, MSE=0.015498
EEpoch 3/10, MSE=0.013544
EEpoch 4/10, MSE=0.014046
EEpoch 5/10, MSE=0.014629
EEpoch 6/10, MSE=0.013470
EEpoch 7/10, MSE=0.016820
EEpoch 8/10, MSE=0.013420
EEpoch 9/10, MSE=0.014642
EEpoch 10/10, MSE=0.014603

--- Training Layer 2/2 ---
EEpoch 1/10, MSE=0.021970
EEpoch 2/10, MSE=0.012198
EEpoch 3/10, MSE=0.009226
EEpoch 4/10, MSE=0.006882
EEpoch 5/10, MSE=0.006127
EEpoch 6/10, MSE=0.006814
EEpoch 7/10, MSE=0.005594
EEpoch 8/10, MSE=0.006014
EEpoch 9/10, MSE=0.006266
EEpoch 10/10, MSE=0.005914
